# Chapter 6 — robust variance components (`var_comprob`)

Reproduces `autism.R` (Example 6.7, Tables 6.8–6.9): a robust linear mixed / variance-component model for autism `vsae` growth across `childid` groups, via `rpm.var_comprob` → `robustvarComp::varComprob`.

`varComprob` is **stochastic** (`lmrob.S` / `TSGS` initials), so we `set_seed` before each fit and check **strict-tier** against direct R.

In [ ]:
import os, sys, pathlib

# Windows R_HOME setup (skip if already configured)
if sys.platform == "win32" and "R_HOME" not in os.environ:
    os.environ["R_HOME"] = r"C:\Program Files\R\R-4.5.2"
    os.environ["PATH"] = r"C:\Program Files\R\R-4.5.2\bin\x64;" + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstattm_py as rpm
from robstattm_py import set_seed
from robstattm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
ro.r("suppressMessages(library(robustvarComp))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## Data preparation (complete cases, 41 children × 5 time-points)

In [ ]:
ro.r("""
suppressMessages(library(robustvarComp))
data(autism, package='WWGbook')
autism <- autism[complete.cases(autism),]
completi <- table(autism$childid)==5
completi <- names(completi[completi])
indici <- as.vector(unlist(sapply(completi, function(x) which(autism$childid==x))))
ind <- rep(FALSE, nrow(autism)); ind[indici] <- TRUE
autism <- subset(autism, subset=ind)
sicdegp <- autism$sicdegp; age <- autism$age
age.2 <- age - 2
sicdegp2 <- sicdegp
sicdegp2[sicdegp == 3] <- 0; sicdegp2[sicdegp == 2] <- 2; sicdegp2[sicdegp == 1] <- 1
sicdegp2.f <- factor(sicdegp2)
autism.updated <- subset(data.frame(autism, sicdegp2.f, age.2), !is.na(vsae))
p <- 5; n <- 41
z1 <- rep(1, p); z2 <- c(0,1,3,7,11); z3 <- z2^2
K <- list(tcrossprod(z1,z1), tcrossprod(z2,z2), tcrossprod(z3,z3),
          tcrossprod(z1,z2)+tcrossprod(z2,z1), tcrossprod(z1,z3)+tcrossprod(z3,z1),
          tcrossprod(z3,z2)+tcrossprod(z2,z3))
names(K) <- c("Int","age","age2","Int:age","Int:age2","age:age2")
groups <- cbind(rep(1:p, each=n), rep((1:n), p))
""")

# Rebuild the prepared frame column-by-column (whole-frame auto-conversion is
# fragile); verified to give a bit-identical model matrix.
vsae = np.asarray(ro.r("as.numeric(autism.updated$vsae)"), float)
age2 = np.asarray(ro.r("as.numeric(autism.updated[[\'age.2\']])"), float)
codes = np.asarray(ro.r("as.integer(autism.updated[[\'sicdegp2.f\']])"), int) - 1
levels = [str(x) for x in ro.r("levels(autism.updated[[\'sicdegp2.f\']])")]
sic = pd.Categorical.from_codes(codes, categories=levels)
autism_df = pd.DataFrame({"vsae": vsae, "age.2": age2, "sicdegp2.f": sic})

# K kernels + groups matrix (exactly as autism.R)
p, n = 5, 41
z1 = np.ones(p); z2 = np.array([0.,1.,3.,7.,11.]); z3 = z2**2
K = [np.outer(z1,z1), np.outer(z2,z2), np.outer(z3,z3),
     np.outer(z1,z2)+np.outer(z2,z1), np.outer(z1,z3)+np.outer(z3,z1),
     np.outer(z3,z2)+np.outer(z2,z3)]
Knames = ("Int","age","age2","Int:age","Int:age2","age:age2")
groups = np.column_stack([np.repeat(np.arange(1,p+1), n), np.tile(np.arange(1,n+1), p)])
FIXED = "vsae ~ age.2 + I(age.2^2) + sicdegp2.f + age.2:sicdegp2.f + I(age.2^2):sicdegp2.f"
print(f"autism: {autism_df.shape[0]} obs, {p} time-points x {n} children")

## Composite Tau estimator (Table 6.8)

Default method (`compositeTau`, `psi='optimal'`) with the lower bounds the script uses.

In [ ]:
ctrl = rpm.var_comprob_control(lower=[0.01, 0.01, 0.01, -np.inf, -np.inf, -np.inf])
set_seed(2468)
ct = rpm.var_comprob(FIXED, autism_df, groups=groups, varcov=K, varcov_names=Knames, control=ctrl)
print(ct)
tau_table = pd.DataFrame({"coef": ct.beta, "SE": np.sqrt(np.diag(ct.vcov_beta))}, index=ct.beta_names)
print("\nFixed effects (Composite Tau):"); print(tau_table.round(4))
print("\nvariance components (eta):", dict(zip(ct.eta_names, np.round(ct.eta, 4))))
print("error variance (sigma2):", round(ct.sigma2, 5))

# strict-tier check vs direct R
ro.r("ctrlR <- varComprob.control(lower=c(0.01,0.01,0.01,-Inf,-Inf,-Inf))")
ro.r(f"set.seed(2468L); rCT <- varComprob({FIXED}, groups=groups, data=autism.updated, varcov=K, control=ctrlR)")
assert np.array_equal(ct.beta, np.asarray(ro.r("as.numeric(rCT$beta)"), float))
assert np.array_equal(ct.eta, np.asarray(ro.r("as.numeric(rCT$eta)"), float))
assert ct.sigma2 == float(ro.r("as.numeric(rCT$sigma2)")[0])
print("\nstrict-tier vs R: OK")

## Classic S estimator (Table 6.9)

`method='S'`, `psi='rocke'`, `cov.init='covOGK'`.

In [ ]:
ctrlS = rpm.var_comprob_control(method="S", psi="rocke", cov_init="covOGK",
                                lower=[0.01, 0.01, 0.01, -np.inf, -np.inf, -np.inf])
set_seed(2468)
cs = rpm.var_comprob(FIXED, autism_df, groups=groups, varcov=K, varcov_names=Knames, control=ctrlS)
print(cs)
s_table = pd.DataFrame({"coef": cs.beta, "SE": np.sqrt(np.diag(cs.vcov_beta))}, index=cs.beta_names)
print("\nFixed effects (Classic S):"); print(s_table.round(4))
ro.r("ctrlSR <- varComprob.control(method='S', psi='rocke', cov.init='covOGK', lower=c(0.01,0.01,0.01,-Inf,-Inf,-Inf))")
ro.r(f"set.seed(2468L); rS <- varComprob({FIXED}, groups=groups, data=autism.updated, varcov=K, control=ctrlSR)")
assert np.array_equal(cs.beta, np.asarray(ro.r("as.numeric(rS$beta)"), float))
print("\nstrict-tier vs R: OK — autism.R reproduced.")